# Hermes Agent no Google Colab ☤

Este notebook clona este repositório no runtime do Colab, instala o Hermes Agent em um ambiente virtual isolado e executa uma primeira consulta usando o modo **one-shot** (`hermes -z`).

> **Por que `-z`?** O TUI interativo do Hermes precisa de um terminal real, enquanto as células do Colab não fornecem um TTY. O modo one-shot funciona bem em notebooks e ainda mantém as ferramentas, skills e o acesso ao diretório do repositório.

## Antes de começar

1. Em **Runtime → Run all**, selecione uma versão recente do Python (3.11–3.13).
2. Crie um segredo do Colab chamado `OPENROUTER_API_KEY` ou digite a chave com segurança quando a célula de autenticação solicitar. A chave não é salva no notebook nem exibida.
3. Altere `REPO_URL`, `REPO_REF` ou `MODEL` nas células abaixo se quiser usar um fork, branch ou modelo diferente.

O runtime do Colab é temporário. Arquivos e configuração criados nele desaparecem quando a sessão termina, a menos que você os copie para o Google Drive.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# Repositório que será executado no runtime do Colab.
REPO_URL = "https://github.com/anonyby777-lgtm/hermes-agent.git"
REPO_REF = "main"
REPO_DIR = Path("/content/hermes-agent")

def run(command, *, cwd=None, env=None):
    """Run a command while preserving useful Colab output."""
    print("$", " ".join(str(part) for part in command))
    return subprocess.run(command, cwd=cwd, env=env, check=True)

if (REPO_DIR / ".git").is_dir():
    print(f"Reutilizando o repositório existente em {REPO_DIR}")
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("Commit carregado:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("Diretório de trabalho:", Path.cwd())

## Instalação

A instalação abaixo usa o `uv.lock` do repositório (`--locked`) e instala apenas as dependências base, suficientes para conversar com o agente e usar os toolsets de arquivos e terminal. Isso evita instalar integrações opcionais desnecessárias no runtime gratuito do Colab.

Para uma instalação completa, defina `INSTALL_ALL_EXTRAS = True`; ela também instala integrações opcionais como dashboard, MCP e Google Workspace.

In [ ]:
# O pacote uv é instalado no Python do runtime apenas se ainda não estiver disponível.
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

UV = shutil.which("uv") or "uv"
VENV_DIR = REPO_DIR / ".venv"
install_args = [UV, "sync", "--locked"]
INSTALL_ALL_EXTRAS = False
if INSTALL_ALL_EXTRAS:
    install_args += ["--extra", "all"]

install_env = os.environ.copy()
install_env["UV_PROJECT_ENVIRONMENT"] = str(VENV_DIR)
run(install_args, cwd=REPO_DIR, env=install_env)

HERMES = VENV_DIR / ("Scripts/hermes.exe" if os.name == "nt" else "bin/hermes")
if not HERMES.exists():
    raise FileNotFoundError(f"Executável do Hermes não encontrado: {HERMES}")

# Mantém toda a configuração da sessão dentro do runtime efêmero do Colab.
os.environ["HERMES_HOME"] = "/content/hermes-home"
Path(os.environ["HERMES_HOME"]).mkdir(parents=True, exist_ok=True)
print("Hermes instalado em:", HERMES)
print(subprocess.check_output([str(HERMES), "--version"], text=True).strip())

## Configuração do modelo

O exemplo usa o OpenRouter porque ele permite escolher entre vários modelos com uma única chave. Você pode trocar `PROVIDER`, `MODEL` e `API_KEY_ENV` por outro provedor suportado pelo Hermes.

Nunca cole uma chave diretamente no código de um notebook compartilhado. Prefira **Colab → Secrets**; o fallback abaixo usa `getpass` e não mostra o valor na saída.

In [ ]:
from getpass import getpass

PROVIDER = "openrouter"
MODEL = "openai/gpt-4o-mini"  # Troque por qualquer modelo disponível no seu provedor.
API_KEY_ENV = "OPENROUTER_API_KEY"

def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

api_key = colab_secret(API_KEY_ENV)
if not api_key:
    api_key = getpass(f"Digite {API_KEY_ENV} (a entrada ficará oculta): " ).strip()
if not api_key:
    raise ValueError("Nenhuma chave de API foi configurada.")

os.environ[API_KEY_ENV] = api_key
print(f"Credencial carregada via {API_KEY_ENV}; o valor não será exibido.")

## Primeira execução

`TOOLSETS` limita as capacidades disponíveis nesta execução. O modo `-z` aprova automaticamente comandos de ferramenta porque não há como responder a um prompt interativo; revise o prompt antes de executar e use um runtime descartável do Colab.

In [ ]:
TOOLSETS = "file,terminal"
USAGE_FILE = Path("/content/hermes-usage.json")

def ask_hermes(prompt, *, model=MODEL, provider=PROVIDER, toolsets=TOOLSETS):
    """Send one prompt to Hermes and return only its final response."""
    command = [str(HERMES), "-z", prompt, "--usage-file", str(USAGE_FILE)]
    if provider:
        command += ["--provider", provider]
    if model:
        command += ["--model", model]
    if toolsets:
        command += ["--toolsets", toolsets]

    execution_env = os.environ.copy()
    execution_env["HERMES_HOME"] = os.environ["HERMES_HOME"]
    result = subprocess.run(
        command,
        cwd=REPO_DIR,
        env=execution_env,
        text=True,
        capture_output=True,
    )
    if result.returncode:
        print(result.stderr or "Hermes encerrou sem uma mensagem de erro.")
        raise RuntimeError(f"Hermes retornou código {result.returncode}.")
    print(result.stdout)
    return result.stdout

answer = ask_hermes(
    "Leia o repositório atual. Resuma em português o que o Hermes Agent faz,"
    " cite os principais diretórios e sugira três formas seguras de começar a explorá-lo."
)

## Fazer outra pergunta

Edite `prompt` ou use `input()` para enviar novas consultas ao mesmo runtime. Cada chamada é uma execução independente; a configuração e o histórico do Colab continuam disponíveis enquanto a sessão estiver ativa.

In [ ]:
prompt = input("Pergunta para o Hermes (deixe vazio para pular): " ).strip()
if prompt:
    ask_hermes(prompt)

## Diagnóstico de consumo

O arquivo de uso é sobrescrito a cada execução e contém tokens, modelo, provedor e custo estimado quando o provedor fornece esses dados.

In [ ]:
import json

if USAGE_FILE.exists():
    usage = json.loads(USAGE_FILE.read_text())
    for key in ("model", "provider", "total_tokens", "estimated_cost_usd", "completed", "failed"):
        print(f"{key}: {usage.get(key)}")
else:
    print("Ainda não há um relatório de uso.")